# Controles complementares (que exigem rodar/ler os modelos)

Estes são os dois controles que **não** saem do CSV agregado e precisam dos modelos
(`transformers` + acesso ao HuggingFace Hub):

1. **Tokenização** — sobreposição de vocabulário (Jaccard) e fertilidade de subwords entre
   tokenizadores, com foco no par **pt–gl**.
2. **Modelo (mesma língua, 2 modelos)** — correlaciona dois CSVs gerados por modelos
   diferentes da *mesma* língua (índice, matched-LOO e camada).

> Os controles de **corpus** (train vs test) e de **modelo mono vs mBERT** já foram
> respondidos direto do CSV agregado; não estão aqui.

## 0. Imports e configuração

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

DEPS = ["nsubj", "obj", "case", "amod"]

# Modelos monolíngues por língua (para o controle de tokenização)
MODELS = {
    "pt": "neuralmind/bert-base-portuguese-cased",
    "gl": "marcosgg/bert-base-gl-cased",
    "es": "dccuchile/bert-base-spanish-wwm-cased",
    "it": "dbmdz/bert-base-italian-xxl-cased",
    "ro": "dumitrescustefan/bert-base-romanian-cased-v1",
    "fr": "camembert-base",
    "de": "bert-base-german-cased",
}

## 1. Controle de tokenização

Carrega os tokenizadores e mede (i) o **Jaccard** entre vocabulários e (ii) a **fertilidade**
(subwords por palavra) numa frase comum. Requer `transformers` e download dos tokenizadores.

In [2]:
def tokenization_overlap(models=MODELS,
                         sentence="O gato preto dormiu na cadeira velha da cozinha ."):
    from transformers import AutoTokenizer
    toks = {lg: AutoTokenizer.from_pretrained(m) for lg, m in models.items()}
    vocabs = {lg: set(t.get_vocab().keys()) for lg, t in toks.items()}

    print("=== Jaccard de vocabulário entre tokenizadores ===")
    langs = list(models)
    jacc = {}
    for i in range(len(langs)):
        for j in range(i + 1, len(langs)):
            a, b = langs[i], langs[j]
            jac = len(vocabs[a] & vocabs[b]) / len(vocabs[a] | vocabs[b])
            jacc[(a, b)] = jac
            print(f"   {a}-{b}: {jac:.3f}")

    print("\n=== Fertilidade (subwords/token) numa frase comum ===")
    for lg, t in toks.items():
        n_words = len(sentence.split())
        n_sub = len(t.tokenize(sentence))
        print(f"   {lg}: {n_sub/n_words:.2f}  ({n_sub} subwords / {n_words} palavras)")
    return jacc

In [3]:
# Executa o controle de tokenização (precisa de transformers + internet)
jacc = tokenization_overlap()

=== Jaccard de vocabulário entre tokenizadores ===
   pt-gl: 0.118
   pt-es: 0.188
   pt-it: 0.130
   pt-ro: 0.094
   pt-fr: 0.025
   pt-de: 0.079
   gl-es: 0.167
   gl-it: 0.078
   gl-ro: 0.082
   gl-fr: 0.021
   gl-de: 0.046
   es-it: 0.136
   es-ro: 0.104
   es-fr: 0.029
   es-de: 0.085
   it-ro: 0.117
   it-fr: 0.031
   it-de: 0.082
   ro-fr: 0.031
   ro-de: 0.069
   fr-de: 0.019

=== Fertilidade (subwords/token) numa frase comum ===
   pt: 1.20  (12 subwords / 10 palavras)
   gl: 1.40  (14 subwords / 10 palavras)
   es: 1.80  (18 subwords / 10 palavras)
   it: 1.70  (17 subwords / 10 palavras)
   ro: 1.90  (19 subwords / 10 palavras)
   fr: 2.10  (21 subwords / 10 palavras)
   de: 2.20  (22 subwords / 10 palavras)


**Interpretação.** Se a proximidade de tokenização (Jaccard) prediz o $\rho$ tão bem quanto
a proximidade tipológica, então tokenização e língua estão confundidas. Para o par **pt–gl**,
um Jaccard alto explicaria parte do $\rho \approx 0.9$.

## 2. Controle de modelo — mesma língua, dois modelos

Primeiro gere os CSVs rodando o pipeline com **cada modelo separadamente** (no terminal):

```bash
python src/run_ud_attention_eval.py --langs pt --splits test \
    --model_mode manual_unico \
    --manual_model neuralmind/bert-base-portuguese-cased \
    --out_csv data/outputs/pt_base.csv

python src/run_ud_attention_eval.py --langs pt --splits test \
    --model_mode manual_unico \
    --manual_model neuralmind/bert-large-portuguese-cased \
    --out_csv data/outputs/pt_large.csv
# idem para gl: marcosgg/bert-base-gl-cased  vs  fpuentes/bert-galician
```

> **Atenção:** `bert-large` tem **24 camadas × 16 cabeças**. Para comparar com um `base`,
> restrinja a uma sub-grade comum (12×12) ou compare só no nível macro. A função abaixo
> exige que ambos os CSVs tenham a **mesma grade**.

In [13]:
import numpy as np, pandas as pd
from scipy.optimize import linear_sum_assignment

DEPS = ["nsubj", "obj", "case", "amod"]

def _spear(a, b):
    ra = pd.Series(a).rank().to_numpy(); rb = pd.Series(b).rank().to_numpy()
    return float(np.corrcoef(ra, rb)[0, 1]) if np.std(ra) * np.std(rb) > 0 else np.nan

def _grid(df):
    return int(df["layer"].max()), int(df["head"].max())

def _mat(df, dep, L, H):
    M = np.zeros((L, H)); s = df[df.deprel == dep]
    for _, r in s.iterrows():
        M[int(r["layer"]) - 1, int(r["head"]) - 1] = float(r["mean_attention"])
    return M

def _layer_profile(df, dep, L, H):
    return _mat(df, dep, L, H).mean(axis=1)

def _resample(u, n):
    L = len(u)
    return np.interp(np.linspace(0, 1, n), np.linspace(0, 1, L), u)

def two_models_same_language(csv_a, csv_b, direction="head_to_dep", macro_points=12):
    a = pd.read_csv(csv_a); a = a[a.direction == direction]
    b = pd.read_csv(csv_b); b = b[b.direction == direction]
    La, Ha = _grid(a); Lb, Hb = _grid(b)

    if (La, Ha) == (Lb, Hb):
        L, H = La, Ha
        SA = np.stack([_mat(a, d, L, H) for d in DEPS], axis=2)
        SB = np.stack([_mat(b, d, L, H) for d in DEPS], axis=2)
        print(f"Grades iguais ({L}x{H}). Analise completa:")
        print(f"{'rel':6} {'index':>8} {'matched':>8} {'layer':>8}")
        for di, dep in enumerate(DEPS):
            idx = _spear(SA[:, :, di].flatten(), SB[:, :, di].flatten())
            keep = [j for j in range(len(DEPS)) if j != di]; bb = np.zeros((L, H))
            for l in range(L):
                C = np.linalg.norm(SA[l][:, keep][:, None, :] - SB[l][:, keep][None, :, :], axis=2)
                _, ci = linear_sum_assignment(C); bb[l] = SB[l, ci, di]
            mt = _spear(SA[:, :, di].flatten(), bb.flatten())
            ly = _spear(SA[:, :, di].mean(axis=1), SB[:, :, di].mean(axis=1))
            print(f"{dep:6} {idx:8.3f} {mt:8.3f} {ly:8.3f}")
    else:
        print(f"Grades DIFERENTES ({La}x{Ha} vs {Lb}x{Hb}).")
        print(f"Index/matched nao se aplicam (no de cabecas difere). So macro,")
        print(f"reamostrado por profundidade relativa para {macro_points} pontos:")
        print(f"{'rel':6} {'macro_resampled':>16}")
        for dep in DEPS:
            ua = _resample(_layer_profile(a, dep, La, Ha), macro_points)
            ub = _resample(_layer_profile(b, dep, Lb, Hb), macro_points)
            print(f"{dep:6} {_spear(ua, ub):16.3f}")

In [7]:
# Gera data/outputs/pt_base.csv  (requer transformers/torch instalados no ambiente do repo)
REPO_DIR = "/home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)"  # ajuste para o caminho do repositório, ex.: "/caminho/cross-lingual-syntactic-attention"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[GPU] NVIDIA GeForce RTX 3050 6GB Laptop GPU (5.7 GB)
[RUN] lang=pt split=test family=manual model=neuralmind/bert-base-portuguese-cased
  [MODEL] neuralmind/bert-base-portuguese-cased carregado em cuda
[OK] CSV agregado: /home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)/data/outputs/pt_base.csv


In [ ]:
!mkdir -p "{REPO_DIR}/data/outputs"
!cd "{REPO_DIR}" && python src/run_ud_attention_eval.py --langs pt --splits test --model_mode manual_unico --manual_model neuralmind/bert-base-portuguese-cased --out_csv data/outputs/pt_base.csv

In [8]:
!mkdir -p "{REPO_DIR}/data/outputs"
!cd "{REPO_DIR}" && python src/run_ud_attention_eval.py --langs pt --splits test --model_mode manual_unico --manual_model neuralmind/bert-large-portuguese-cased --out_csv data/outputs/pt_large.csv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[GPU] NVIDIA GeForce RTX 3050 6GB Laptop GPU (5.7 GB)
[RUN] lang=pt split=test family=manual model=neuralmind/bert-large-portuguese-cased
  [MODEL] neuralmind/bert-large-portuguese-cased carregado em cuda
[OK] CSV agregado: /home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)/data/outputs/pt_large.csv


In [15]:
two_models_same_language(
    "/home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)/data/outputs/pt_base.csv",
    "/home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)/data/outputs/pt_large.csv",
)

Grades DIFERENTES (12x12 vs 24x16).
Index/matched nao se aplicam (no de cabecas difere). So macro,
reamostrado por profundidade relativa para 12 pontos:
rel     macro_resampled
nsubj             0.510
obj               0.231
case              0.238
amod             -0.077


**Interpretação.** Um `matched` alto entre dois modelos da **mesma língua** mostra que
cabeças funcionalmente equivalentes existem e são recuperáveis — separando o *efeito de
instância do modelo* da *propriedade linguística*.

In [16]:
REPO_DIR = "/home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)"

!mkdir -p "{REPO_DIR}/data/outputs"
!cd "{REPO_DIR}" && python src/run_ud_attention_eval.py --langs gl --splits test --model_mode manual_unico --manual_model marcosgg/bert-base-gl-cased --out_csv data/outputs/gl_marcosgg.csv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[GPU] NVIDIA GeForce RTX 3050 6GB Laptop GPU (5.7 GB)
[RUN] lang=gl split=test family=manual model=marcosgg/bert-base-gl-cased
  [MODEL] marcosgg/bert-base-gl-cased carregado em cuda
[OK] CSV agregado: /home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)/data/outputs/gl_marcosgg.csv


In [17]:
!cd "{REPO_DIR}" && python src/run_ud_attention_eval.py --langs gl --splits test --model_mode manual_unico --manual_model fpuentes/bert-galician --out_csv data/outputs/gl_fpuentes.csv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[GPU] NVIDIA GeForce RTX 3050 6GB Laptop GPU (5.7 GB)
[RUN] lang=gl split=test family=manual model=fpuentes/bert-galician
tokenizer_config.json: 100%|███████████████████| 570/570 [00:00<00:00, 3.30MB/s]
vocab.txt: 242kB [00:00, 2.24MB/s]
tokenizer.json: 729kB [00:00, 6.71MB/s]
config.json: 100%|█████████████████████████████| 715/715 [00:00<00:00, 3.21MB/s]
pytorch_model.bin: 100%|█████████████████████| 439M/439M [01:55<00:00, 3.79MB/s]
  [MODEL] fpuentes/bert-galician carregado em cuda
[OK] CSV agregado: /home/ricardo/doutorado/SEPLN - Linguas_Romanicas/git/cross-lingual-syntactic-attention-main (2)/data/outputs/gl_fpuentes.csv
model.safetensors: 100%|█████████████████████| 439M/439M [02:04<00:00, 3.52MB/s]


In [18]:
two_models_same_language(
    f"{REPO_DIR}/data/outputs/gl_marcosgg.csv",
    f"{REPO_DIR}/data/outputs/gl_fpuentes.csv",
)

Grades iguais (12x12). Analise completa:
rel       index  matched    layer
nsubj     0.140    0.655    0.699
obj       0.160    0.752    0.748
case     -0.026    0.443    0.490
amod      0.011    0.653    0.769
